In [30]:
# %pip install yahoo_oauth yahoo_fantasy_api pandas scikit-learn matplotlib seaborn openai requests
%pip install yfpy pandas scikit-learn matplotlib seaborn openai requests dotenv supabase


  Obtaining dependency information for supabase from https://files.pythonhosted.org/packages/95/06/5e6f4bf89dedadf81f893832115c1866da1aa093142c9989c2818780dae3/supabase-2.31.0-py3-none-any.whl.metadata
  Obtaining dependency information for realtime==2.31.0 from https://files.pythonhosted.org/packages/13/60/164246615e8b059f6d53d34648a0784260421ec98a07eb1e45160f063221/realtime-2.31.0-py3-none-any.whl.metadata
  Obtaining dependency information for supabase-functions==2.31.0 from https://files.pythonhosted.org/packages/90/79/1a8162ce7d705381a4f2668c0da68e097b103c1aa701418a31da52905c7b/supabase_functions-2.31.0-py3-none-any.whl.metadata
  Obtaining dependency information for storage3==2.31.0 from https://files.pythonhosted.org/packages/2f/b2/60d86a3a99ae743e8a00a6df912f85269b3bc882ba217b8ce1690881c669/storage3-2.31.0-py3-none-any.whl.metadata
  Obtaining dependency information for supabase-auth==2.31.0 from https://files.pythonhosted.org/packages/ca/5e/22f3b0546bb1f0985f06fb1ebf5f6405a3b1

In [3]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# from yahoo_oauth import OAuth2
# from yahoo_fantasy_api import league
from sklearn.linear_model import LinearRegression
from openai import OpenAI
import requests

sns.set_style("whitegrid")


In [4]:

from yfpy.query import YahooFantasySportsQuery
from pathlib import Path

# Yahoo league ID dictionary
yahoo_league_ids = {
    2025: "49987", 2024: "17888", 2023: "5559", 2022: "30577",
    2021: "39016", 2020: "350814", 2019: "446669", 2018: "137260",
    2017: "565769", 2016: "182400", 2015: "137181"
    }

league_history = {}
for key, value in yahoo_league_ids.items():
    league_history[key] = {'League ID':value}

# First time setup (browser will open for authorization)
# The library handles the OAuth2 flow and stores the tokens
# It will generate a 'token.json' file after successful authorization
# which it will reuse in subsequent runs
yahoo_auth_dict = json.load(open('yahoo_auth.json','r'))

# print(dict)
league = YahooFantasySportsQuery(
    league_id="49987",  # Replace with your league ID
    game_code="nfl", # Or use game_id if you prefer (e.g., 449 for NFL 2023)
    offline=False,
    yahoo_consumer_key= yahoo_auth_dict['yahoo_consumer_key'],
    yahoo_consumer_secret=yahoo_auth_dict['yahoo_consumer_secret'],
    save_token_data_to_env_file=True,
    env_file_location = Path('/Users/anthony/Desktop/VSFolder/sports_analytics/')
)
# Getting league info to split out by year
league_hist_meta_data = league.get_user_games()

league_history


{2025: {'League ID': '49987'},
 2024: {'League ID': '17888'},
 2023: {'League ID': '5559'},
 2022: {'League ID': '30577'},
 2021: {'League ID': '39016'},
 2020: {'League ID': '350814'},
 2019: {'League ID': '446669'},
 2018: {'League ID': '137260'},
 2017: {'League ID': '565769'},
 2016: {'League ID': '182400'},
 2015: {'League ID': '137181'}}

In [5]:
league_history[2022]

{'League ID': '30577'}

In [6]:
for l in league_hist_meta_data:
    if l.name == 'Football' and l.season in league_history.keys():
        league_season = YahooFantasySportsQuery(
            league_id=league_history[l.season]["League ID"],  # Replace with your league ID
            game_code="nfl", # Or use game_id if you prefer (e.g., 449 for NFL 2023)
            game_id= l.game_id,  # Use the game_id from the metadata
            offline=False,
            yahoo_consumer_key= yahoo_auth_dict['yahoo_consumer_key'],
            yahoo_consumer_secret=yahoo_auth_dict['yahoo_consumer_secret'],
            save_token_data_to_env_file=True,
            env_file_location = Path('/Users/anthony/Desktop/VSFolder/sports_analytics/')
        )
        league_history[l.season] = {**league_history[l.season], **{'Game ID':l.game_id,'Season':league_season}} # game_id = game_key

In [8]:
league_history[2022]['Season'].get_league_standings().teams[0].points_for

1353.24

In [9]:
# Run this for each season
league_team_df = pd.DataFrame()
for season in league_history.keys():
    teams = league_history[season]['Season'].get_league_standings().teams

    for team_num in range(len(teams)-1):
        team_df = pd.DataFrame()

        # Team and manager info
        team_df.loc[0,"Season"] = teams[team_num].team_points.season
        team_df.loc[0,"Team Name"] = teams[team_num].name.upper().decode('utf-8',errors='ignore')
        # print(type(teams[team_num].name))
        team_df.loc[0,"Team ID"] = teams[team_num].team_id
        team_df.loc[0,"Team Key"] = teams[team_num].team_key
        team_df.loc[0,"Manager Name"] = teams[team_num].managers[0].nickname.upper()
        team_df.loc[0,"Manager ID"] = teams[team_num].managers[0].manager_id
        team_df.loc[0,"Tier"] = teams[team_num].managers[0].felo_tier.upper()
        
        # Draft and move data
        team_df.loc[0,"Draft Grade"] = teams[team_num].draft_grade
        team_df.loc[0,"FaaB Balance"] = teams[team_num].faab_balance
        team_df.loc[0,"Total Moves"] = teams[team_num].number_of_moves
        team_df.loc[0,"Total Trades"] = teams[team_num].number_of_trades

        # Standings and points data
        team_df.loc[0,"W"] = teams[team_num].team_standings.outcome_totals.wins
        team_df.loc[0,"L"] = teams[team_num].team_standings.outcome_totals.losses
        team_df.loc[0,"T"] = teams[team_num].team_standings.outcome_totals.ties
        team_df.loc[0,"PCT"] = teams[team_num].team_standings.outcome_totals.percentage
        team_df.loc[0,"Total Points"] = teams[team_num].team_points.total
        team_df.loc[0,"PF"] = teams[team_num].points_for
        team_df.loc[0,"PA"] = teams[team_num].points_against
        team_df.loc[0,"PDiff"] = teams[team_num].points_for - teams[team_num].points_against
        team_df.loc[0,"Rank"] = teams[team_num].rank
        team_df.loc[0,"Clinched Playoffs"] = teams[team_num].clinched_playoffs
        team_df.loc[0,"Playoff Seed"] = teams[team_num].playoff_seed

        league_team_df = pd.concat([league_team_df, team_df], ignore_index=True)
        # print(team_df.head(3))

# Setting data types
league_team_df = league_team_df.astype({'Season':'int','Team ID':'int','Manager ID':'int','Total Moves':'int','Total Trades':'int','W':'int','L':'int','T':'int','Rank':'int','Clinched Playoffs':'int','Playoff Seed':'int'})
league_team_df = league_team_df.sort_values(by=['Season','Rank']).reset_index(drop=True)
league_team_df['FaaB Balance'] = league_team_df['FaaB Balance'].fillna(0).astype('int')



In [10]:
league_team_df.sample(50)

,Season,Team Name,Team ID,Team Key,Manager Name,Manager ID,Tier,Draft Grade,FaaB Balance,Total Moves,...,L,T,PCT,Total Points,PF,PA,PDiff,Rank,Clinched Playoffs,Playoff Seed
64,2020,GREG FOCKER,7,399.l.350814.t.7,JOHNNY,7,GOLD,B,0,11,...,5,0,0.615,1266.68,1266.68,1139.38,127.30,4,1,3
118,2025,STAIRWAY TO SEVEN,9,461.l.49987.t.9,KEVIN,9,SILVER,B+,10,28,...,5,0,0.643,1392.56,1392.56,1281.08,111.48,3,1,4
117,2025,BUSINESS ETHICS,6,461.l.49987.t.6,CORY,6,GOLD,C,55,12,...,3,0,0.786,1398.20,1398.20,1169.12,229.08,2,1,1
77,2021,BUSINESS ETHICS,5,406.l.39016.t.5,CORY,5,SILVER,D,42,32,...,6,0,0.571,1436.26,1436.26,1380.88,55.38,6,1,4
73,2021,GREG FOCKER,6,406.l.39016.t.6,JOHNNY,6,PLATINUM,B,23,6,...,5,0,0.643,1432.58,1432.58,1179.28,253.30,2,1,3
46,2018,BLITZKRIEG,4,380.l.137260.t.4,EDDIE,4,SILVER,C+,0,18,...,10,0,0.231,1152.82,1152.82,1358.42,-205.60,8,0,10
16,2016,TRUFFLE BUTTER,8,359.l.182400.t.8,MATT,8,BRONZE,B,0,30,...,5,0,0.615,1274.98,1274.98,1204.70,70.28,4,1,3
111,2024,BUSINESS ETHICS,6,449.l.17888.t.6,CORY,6,SILVER,A+,37,20,...,7,0,0.500,1276.08,1276.08,1235.46,40.62,7,0,7
124,2025,AT LEAST IM NOT LAST,2,461.l.49987.t.2,JP,2,BRONZE,A,8,37,...,8,0,0.429,1168.48,1168.48,1188.44,-19.96,9,0,8
53,2019,BUSINESS ETHICS,6,390.l.446669.t.6,CORY,6,GOLD,A,0,16,...,3,0,0.769,1263.62,1263.62,1111.48,152.14,4,1,1


In [ ]:
# Pushing league team data to Supabase

# def to_supabase_table():
from supabase import create_client
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

SUPABASE_URL = "https://your-project.supabase.co"
SUPABASE_KEY = "your-anon-or-service-role-key"  # Use service role key for writes

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)


In [15]:
league_history.items()

dict_items([(2025, {'League ID': '49987', 'Game ID': 461, 'Season': <yfpy.query.YahooFantasySportsQuery object at 0x12c09b050>}), (2024, {'League ID': '17888', 'Game ID': 449, 'Season': <yfpy.query.YahooFantasySportsQuery object at 0x12c00ee90>}), (2023, {'League ID': '5559', 'Game ID': 423, 'Season': <yfpy.query.YahooFantasySportsQuery object at 0x12c07f810>}), (2022, {'League ID': '30577', 'Game ID': 414, 'Season': <yfpy.query.YahooFantasySportsQuery object at 0x12c05ca10>}), (2021, {'League ID': '39016', 'Game ID': 406, 'Season': <yfpy.query.YahooFantasySportsQuery object at 0x12bf04b50>}), (2020, {'League ID': '350814', 'Game ID': 399, 'Season': <yfpy.query.YahooFantasySportsQuery object at 0x12bf312d0>}), (2019, {'League ID': '446669', 'Game ID': 390, 'Season': <yfpy.query.YahooFantasySportsQuery object at 0x12c05e310>}), (2018, {'League ID': '137260', 'Game ID': 380, 'Season': <yfpy.query.YahooFantasySportsQuery object at 0x12bf26510>}), (2017, {'League ID': '565769', 'Game ID': 

In [11]:
def get_league_standings(league_history) -> pd.DataFrame:
    standings_list = []
    for year, data in league_history.items():
        season = data['Season']
        standings = season.get_league_standings()
        for team in standings:
            team_data = {
                'Year': year,
                'Team Name': team['name'],
                'Wins': team['wins'],
                'Losses': team['losses'],
                'Ties': team['ties'],
                'Points For': team['points_for'],
                'Points Against': team['points_against']
            }
            standings_list.append(team_data)
    
    return pd.DataFrame(standings_list)



In [19]:
draft_25 = league_history[2025]['Season'].get_league_draft_results()

In [22]:
league_history[2022]

{'League ID': '30577',
 'Game ID': 414,
 'Season': <yfpy.query.YahooFantasySportsQuery at 0x12c05ca10>}

In [21]:
#### Figuring out what is produced through Yahoo functions
type(draft_25), len(draft_25)

(list, 180)

In [1]:
league_history[2025]['Draft Results'][0]

NameError: name 'league_history' is not defined

In [12]:
## Adding to values to the sub-dictionary to query by season
for s in league_history.keys():
    league_history[s]['Draft Results'] = league_history[s]['Season'].get_league_draft_results()
    league_history[s]['Match Ups By Week'] = league_history[s]['Season'].get_league_matchups_by_week()
    league_history[s]['Players'] = league_history[s]['Season'].get_league_players()
    league_history[s]['Scoreboard By Week'] = league_history[s]['Season'].get_league_scoreboard_by_week()
    league_history[s]['Standings'] = league_history[s]['Season'].get_league_standings()
    league_history[s]['Player Draft Analysis'] = league_history[s]['Season'].get_player_draft_analysis()
    league_history[s]['Transaction History'] = league_history[s]['Season'].get_league_transactions()
    # Player percent owned by week
    league_history[s]['Player Percent Owned By Week'] = league_history[s]['Season'].get_player_percent_owned_by_week()
    # Player stats by date
    league_history[s]['Player Stats By Date'] = league_history[s]['Season'].get_player_stats_by_date()
    # Player stats by week
    league_history[s]['Player Stats By Week'] = league_history[s]['Season'].get_player_stats_by_week()
    # Player stats by season
    league_history[s]['Player Stats By Season'] = league_history[s]['Season'].get_player_stats_for_season()
    # Get team standings
    league_history[s]['Team Standings'] = league_history[s]['Season'].get_team_standings()
    # Get team rosters by week
    league_history[s]['Team Rosters By Week'] = league_history[s]['Season'].get_team_roster_by_week()
    # Get team roster player stats by week
    league_history[s]['Team Roster Player Stats By Week'] = league_history[s]['Season'].get_team_roster_player_stats_by_week()
    # Team stats by week
    league_history[s]['Team Stats By Week'] = league_history[s]['Season'].get_team_stats_by_week()
    # Team stats
    league_history[s]['Team Stats By Season'] = league_history[s]['Season'].get_team_stats()

TypeError: 'builtin_function_or_method' object is not iterable

In [34]:
info = league.get_league_info()
print(info)

2025-08-11 21:32:57.985 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.


League({
  "allow_add_to_dl_extra_pos": 1,
  "current_week": 1,
  "draft_results": null,
  "draft_status": "predraft",
  "edit_key": 1,
  "end_date": "2025-12-29",
  "end_week": 17,
  "felo_tier": "silver",
  "game_code": "nfl",
  "iris_group_chat_id": null,
  "is_cash_league": 0,
  "is_plus_league": 0,
  "is_pro_league": 0,
  "league_id": "49987",
  "league_key": "461.l.49987",
  "league_type": "private",
  "league_update_timestamp": null,
  "logo_url": "https://s.yimg.com/ep/cx/blendr/v2/image-avatar-football-trophy-car-png_1721783398928.png",
  "name": "Fantasy Football",
  "num_teams": 12,
  "players": [
    {
      "player": {
        "bye_weeks": {
          "week": 5
        },
        "display_position": "QB",
        "editorial_player_key": "nfl.p.7200",
        "editorial_team_abbr": "Pit",
        "editorial_team_full_name": "Pittsburgh Steelers",
        "editorial_team_key": "nfl.t.23",
        "editorial_team_url": "https://sports.yahoo.com/nfl/teams/pittsburgh/",
       

In [ ]:
# Example: Get league settings
league_settings = league.get_league_settings()
print(dir(league))
# print(f"League Name: {league_settings.name}")
# print(f"League Size: {league_settings.num_teams}")
print(league_settings)
# Example: Get teams in the league
teams = query.get_teams()
for team in teams:
    print(f"Team: {team.name} ({team.team_id})")

# Example: Get players from a specific team (replace with a team ID)
# team_key = "<YOUR_TEAM_KEY>" # You can get this from the teams list
# roster = query.get_roster_by_team_key(team_key, week=1) # Specify week if needed
# for player in roster:
#     print(f"Player: {player.name} ({player.display_position})")

# Example: Convert a list of players to a Pandas DataFrame
# players_data = [{'name': player.name, 'position': player.display_position} for player in roster]
# df = pd.DataFrame(players_data)
# print(df)

In [5]:
# Ensure oauth2.json exists or generate it
# If oauth2.json doesn't exist, create it with your Yahoo credentials
if not os.path.exists("yahoo_oauth2.json"):
    # print("oauth2.json not found. Starting OAuth flow...")
    # CLIENT_ID = input("Enter your Yahoo Client ID: ").strip()
    # CLIENT_SECRET = input("Enter your Yahoo Client Secret: ").strip()

    oauth = OAuth2(CLIENT_ID, CLIENT_SECRET, from_file='yahoo_oauth2.json')
    if not oauth.token_is_valid():
        oauth.refresh_access_token()

    # Save credentials to oauth2.json
    with open("oauth2.json", "w") as f:
        import json
        json.dump(oauth.credentials, f, indent=2)
    print("oauth2.json created successfully!")

else:
    print("Loading existing OAuth credentials from oauth2.json...")
    oauth = OAuth2(None, None, from_file="oauth2.json")

# Verify we have a valid token
if not oauth.token_is_valid():
    oauth.refresh_access_token()
    print("Access token refreshed.")

# Replace with your league ID
league_id = "49987"
try:
    lg = league.League(oauth, league_id)
    standings = lg.standings()
    print("Yahoo API authenticated successfully.")
    USE_API = True
except Exception as e:
    print(f"Yahoo API failed: {e}")
    print("Falling back to CSV exports.")
    USE_API = False


[2025-08-10 14:09:59,252 DEBUG] [yahoo_oauth.oauth.__init__] Checking 


FileNotFoundError: [Errno 2] No such file or directory: 'yahoo_oauth2.json'